# Feature Research Notebook (PR 42)

Training-side research on the accepted feature representations of both cascades. Commit 2 built the
foundation (Cells 1–13): one labelled top-to-bottom path, the cascades' split drawn **training side only**, the
inventory of the accepted features, feature contracts and stage feature populations, and the frozen
references later commits compare against. Commit 3, Step 3A (Cells 14–21) audits the accepted target-derived
encodings: for a row being encoded, which rows and labels reach its value, and which of those labels could
have been known when it was contacted. Step 3B (Cells 22–30) builds past-only versions of the Stage 1
encodings, measures their predictive effect against the accepted ones on identical rows, records the
feature-contract decisions and designs the PR 49 temporal robustness protocol. Each run writes three
write-once research records.

**Rules** (`tests/notebooks/test_notebook_contracts.py`, G5 and G7)

- Training rows only. Cell 2 draws the cascades' split inside `draw_training_side`, which returns the
  training side. No test row, test label, test row identity, test prediction or test metric is bound
  in this notebook.
- Accepted artifacts are opened only through the PR 38 boundary extractors (Cell 11).
- Nothing here saves a model or a cascade artifact, edits production feature code or writes any file except
  this run's records, `research_logs/pr42/<run_id>/feature_inventory.json`, `encoding_audit.json` and
  `temporal_validity.json` (each with its `.sha256`). Step 3B fits Stage 1 research models in memory, on
  training rows, with the accepted PR 39 settings; none is kept.
- Run top to bottom in a fresh kernel, from `feature_research/` or `bank_pipeline/`.

**Sources** (`documentation/notebook_map.md`, section 2.1): **E** = `exploratory_feature_research.ipynb`,
**G** = `glass_cascade.ipynb`, **B** = `blackbox_cascade.ipynb`. Review documents:
`research_logs/pr42/feature_inventory.md` (Commit 2), `research_logs/pr42/encoding_audit.md` (Step 3A) and
`research_logs/pr42/temporal_validity.md` (Step 3B).

## Cell 1 — Imports and path setup
From E 1 and E 3: puts `bank_pipeline/` on `sys.path` (and removes the notebook's own folder, so sibling
modules cannot be imported by bare name), applies the shared seed and plot settings, and imports the
PR 42 research modules.

In [ ]:
# =============================================================================
# CELL 1: PATH BOOTSTRAP, IMPORTS & SETUP
# =============================================================================
import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()

BANK_ROOT = next(
    (p for p in (_here, *_here.parents)
     if (p / "feature_research" / "__init__.py").is_file() and (p / "shared").is_dir()),
    None,
)
if BANK_ROOT is None:
    raise RuntimeError(f"Could not locate bank_pipeline/ above {_here}")

def _is_here(p):
    try:
        return Path(p or ".").resolve() == _here
    except OSError:
        return False

sys.path[:] = [p for p in sys.path if not _is_here(p)]
if str(BANK_ROOT) not in sys.path:
    sys.path.insert(0, str(BANK_ROOT))

from feature_research import RANDOM_SEED, apply_global_settings
from feature_research import encoding_audit as EA
from feature_research import feature_inventory as FI
from feature_research import frozen_references as FR
from feature_research import research_records as RR
from feature_research import temporal_validity as TV

apply_global_settings()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)
print(f"bank_pipeline root: {BANK_ROOT}")

## Cell 2 — Training side of the cascades' split
Replaces E 2, E 4 and E 5. `draw_training_side()` runs the protocol every notebook runs —
`build_database()` → `BankPreprocessor(drop_leaky=True).fit_transform()` → `sanity_check_data` →
`GlobalSplitManager(test_size=0.20, random_state=42).create_split(..., target_col="y")` — inside one call,
and returns only the training rows, their labels, the train-side identities and the PR 38 `TrainReference`
(fingerprints only). The population, the test split and the split manager never leave the call, and the
protocol's own console output (full-population and test-split counts) is discarded.

In [ ]:
# ============================================================
# CELL 2: TRAINING SIDE OF THE CASCADES' SPLIT
# ============================================================
from feature_research.training_side import draw_training_side

TRAIN = draw_training_side()

## Cell 3 — Run identity
A new run id (`c3_<YYYYMMDD_HHMMSS>`), every seed by name, the code identity (HEAD, uncommitted tracked
changes, LF SHA-256 of the files the inventory and the audit trace, whole-tree digest) and the environment. A
run id is used once: Cell 13 creates the run folder and refuses an existing one, and Cells 21 and 30 add the
run's second and third records beside the first. No record is ever written over.

In [ ]:
# ============================================================
# CELL 3: RUN IDENTITY — run id, seeds, code identity
# ============================================================
SEEDS = {"split": RANDOM_SEED, "fold_plans": RANDOM_SEED,
         "target_probe": FI.PROBE_SEED, "parity_folds": RANDOM_SEED,
         "encoding_audit": EA.SEED, "temporal_validity": TV.SEED}
RUN = RR.start_run(commit=3, seeds=SEEDS)
print(RUN.describe())

## Cell 4 — Feature types
From E 6, on the training rows. `classify_features` types an integer column with at most 10 distinct values as
categorical and every other column as numeric, so `job` (12 codes) is numeric and `previous` categorical — as in
E 6's own output, although E 6's markdown table lists those two the other way round.

In [ ]:
# ============================================================
# CELL 4: FEATURE TYPING
# ============================================================
from feature_research import classify_features

NUMERIC_FEATURES, CATEGORICAL_FEATURES = classify_features(TRAIN.frame(), TRAIN.target_col)

## Cell 5 — Stage feature populations
What each stage of each arm consumes, each population checked against the production registry it names
(`LR_FEATURES`, the MLP's `STAGE1_FEATURES`, `RF_SOURCE_FEATURES`, `RF_FEATURES_BINARY`, `EBM_FEATURES`,
`define_ebm_interactions`, `STAGE4_FEATURES`) and against the cascade notebook cells that build or read it
(G 5, G 8, G 8B, G 17–19; B 7, B 9, B 16–18, B 30–31, B 35–36; source only, nothing executed). The research
registries in `feature_research.feature_engineering` are checked too: they are not what the cascades consume.

In [ ]:
# ============================================================
# CELL 5: STAGE FEATURE POPULATIONS — registries and call sites
# ============================================================
POPULATIONS = FI.population_table()
REGISTRY_CHECKS = FI.check_registries()
CALL_SITE_CHECKS = FI.check_call_sites()
print(FI.require(REGISTRY_CHECKS, "registries"))
print(FI.require(CALL_SITE_CHECKS, "call sites"))
POPULATIONS[["key", "model", "n_features", "registry", "contract", "cells"]]

## Cell 6 — Per-feature inventory
One row per (stage, arm, consumed feature): kind, definition, source variables (raw columns, transitively),
target dependence and the fitted features it comes from, fold refits, and the feature's own test-time rule —
stated from the production code, not by reading the test split. The final-fit and cross-validation scopes, and
where they differ, are the population's (`POPULATIONS["fit"]`, Cell 5). Intermediate DAG nodes are in `NODES`.

In [ ]:
# ============================================================
# CELL 6: PER-FEATURE INVENTORY
# ============================================================
INVENTORY = FI.feature_table()
NODES = FI.node_table()
print(INVENTORY.groupby(["stage", "arm"]).size().rename("features").to_string())
INVENTORY[["stage", "arm", "feature", "kind", "target_dependence", "source_variables", "out_of_fit_rule"]]

## Cell 7 — Feature builds on the training rows
From G 5 / B 7 (Stage 1), B 16–17 (Stage 2 sources and bins) and E 11–13 (the feature DAG and its validation),
with the **production** engineers: each fitted on all training rows and applied to them, as the final fits do.
Checks: every declared node is produced by each implementation that claims it, the training-path bins equal the
test-path binning applied to the same rows, every bin family is mutually exclusive and complete, and every
consumed value is finite. `check_nsd_reference` confirms how the density encodes its own fit rows: every production
NSD fit (the final fits and every Stage 2 and Stage 3 fold fit) has more than 10,000 rows, so only a seeded sample
of 5,000 reference rows contributes labels, and the full-train fits of both implementations keep exactly 5,000.

In [ ]:
# ============================================================
# CELL 7: FEATURE BUILDS ON THE TRAINING ROWS
# ============================================================
TRAINING_FRAMES = FI.build_training_frames(TRAIN.X_train, TRAIN.y_train)
DAG_CHECKS = FI.check_dag_columns(TRAINING_FRAMES)
NSD_CHECKS = FI.check_nsd_reference(TRAINING_FRAMES)
print(FI.require(DAG_CHECKS, "DAG columns"))
print(FI.require(NSD_CHECKS, "NSD reference sample"))
print(TRAINING_FRAMES["nsd_fit_sizes"].to_string(index=False))
BUILD_SUMMARY = FI.training_build_summary(TRAINING_FRAMES, TRAIN.y_train)
BUILD_SUMMARY

## Cell 8 — Target dependence, verified
Each implementation is fitted twice on the same training rows, with the true labels and with a seeded
permutation of them, and a fixed sample of those rows is transformed by both fits. A column whose values change
depends on the labels. Every declaration (`none`, `fitted`, `inherited`) must agree; a bin is judged through
its family or its source (a bin is a fixed-threshold function of its source).
This verifies dependence on labels only: whether that dependence is temporally valid is Commit 3.

In [ ]:
# ============================================================
# CELL 8: TARGET-DEPENDENCE PROBE (label permutation)
# ============================================================
PROBE = FI.probe_target_dependence(TRAIN.X_train, TRAIN.y_train)
print(FI.require_probe(PROBE))
PROBE.pivot_table(index=["implementation", "declared"], columns="observed", values="column",
                  aggfunc="count", fill_value=0)

## Cell 9 — Fitting and fold-refit settings
The settings the inventory states, read from the production code: Stage 1 per-fold encoder and scaler refits
and fold counts, the accepted Stage 2 run and its `fold_label_excluded` inner-fold pathway, and both Stage 3
configurations (`per_fold`, 5 tuning and 10 reporting folds, seed 42).

In [ ]:
# ============================================================
# CELL 9: FIT PROTOCOLS
# ============================================================
FIT_CHECKS = FI.check_fit_protocols()
print(FI.require(FIT_CHECKS, "fit protocols"))
FIT_CHECKS[["stage", "check", "passed", "detail"]]

## Cell 10 — Frozen references
Recomputed from this branch and compared with the accepted PR 39 records (read after their `.sha256`
sidecars): the Stage 1 feature contracts, the Stage 2 binning identity, the Stage 3 feature DAG and block-cache
identity, the three EBM interaction pairs, the Stage 4 inputs and the train-side population. `MISMATCH` stops
the notebook; `local verification` items are checked by `tests/realdata/test_pr42_frozen_references.py`.

In [ ]:
# ============================================================
# CELL 10: FROZEN REFERENCES
# ============================================================
FROZEN = pd.DataFrame(FR.frozen_references(TRAIN))
STAGE3_DAG = FR.dag_implementation()
if (FROZEN["status"] == "MISMATCH").any():
    raise AssertionError(FROZEN[FROZEN["status"] == "MISMATCH"].to_string())
print(FROZEN["status"].value_counts().to_string())
FROZEN[["name", "status", "computed", "recorded"]]

## Cell 11 — Accepted artifacts, through the PR 38 boundary extractors only
The frozen Phase 1 Stage 1–3 artifacts (PR 39 Commit 4 manifest), each verified by size and SHA-256 and read
by its boundary extractor, which reads allowlisted training-side fields only. Roles without an extractor (the
black-box MLP and XGBoost) are not opened, and fields the extractors do not expose (such as the EBM's stored
interaction pairs) are not read: Cell 10 states those from the production code. Artifacts are local (`models/` is Git-ignored): an absent file is
recorded as `not available locally`; a changed file or a failed extractor check stops the notebook.

In [ ]:
# ============================================================
# CELL 11: ACCEPTED ARTIFACTS THROUGH THE PR 38 BOUNDARY EXTRACTORS
# ============================================================
ACCEPTED = FR.accepted_artifact_views(TRAIN.train_reference)
ACCEPTED_TABLE = pd.DataFrame(ACCEPTED).reindex(
    columns=["role", "status", "extractor", "checks", "checks_passed", "path"])
if (ACCEPTED_TABLE["status"] == "FAILED").any():
    raise AssertionError(pd.DataFrame(ACCEPTED).to_string())
ACCEPTED_TABLE

## Cell 12 — Duplicated feature-engineering implementations
For the Commit 6 decision (notebook map section 5): the functions `shared/stage2/feature_engineering.py` copies
from `feature_research/feature_engineering/`, their current importers (AST of every Python file and the
version-controlled notebooks), and how far the two implementations — and Stage 1's mirrored features — agree on
the training rows, in-sample and on the shared 5 folds' held-out rows. Values are compared with `compare_values`:
a value missing on one side only counts as a difference, values missing on both sides match, and the maximum
difference is taken over pairs where both values are finite. Nothing is consolidated here.

In [ ]:
# ============================================================
# CELL 12: DUPLICATED FEATURE-ENGINEERING IMPLEMENTATIONS
# ============================================================
DUPLICATES = pd.DataFrame(FI.DUPLICATE_PAIRS)
CALLERS = FI.duplicate_callers()
PARITY, BINNING_PORT = FI.duplicate_parity(TRAIN.X_train, TRAIN.y_train)
print(FI.callers_summary(CALLERS))
print(BINNING_PORT)
PARITY[PARITY["rows_differ"] > 0]

## Cell 13 — Research record: feature inventory
The run's first record: writes `research_logs/pr42/<run_id>/feature_inventory.json` and its `.sha256` sidecar
once (creating the run folder), then reads it back through its sidecar and content digest. `content_sha256`
excludes the run id, times, code and environment, so a reproducing run has the same value. Its sections are the
Commit 2 inventory unchanged; Cell 21 compares them with the verified Commit 2 record.

In [ ]:
# ============================================================
# CELL 13: WRITE THE RESEARCH RECORD
# ============================================================
RECORD_PATH = RR.write_run_record(RUN, "feature_inventory", {
    "scope": "PR 42 Commit 2: feature research foundation and feature inventory (training side)",
    "training_side": {"protocol": dict(TRAIN.protocol), "identities": dict(TRAIN.identities)},
    "feature_typing": {"numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES},
    "populations": POPULATIONS.to_dict("records"),
    "inventory": INVENTORY.to_dict("records"),
    "nodes": NODES.to_dict("records"),
    "checks": {"registries": REGISTRY_CHECKS.to_dict("records"),
               "call_sites": CALL_SITE_CHECKS.to_dict("records"),
               "dag_columns": DAG_CHECKS.to_dict("records"),
               "nsd_reference": NSD_CHECKS.to_dict("records"),
               "fit_protocols": FIT_CHECKS.to_dict("records")},
    "training_frames": BUILD_SUMMARY.to_dict("records"),
    "nsd_reference": {"fit_sizes": TRAINING_FRAMES["nsd_fit_sizes"].to_dict("records"),
                      "full_train": TRAINING_FRAMES["nsd_reference"].to_dict("records")},
    "target_probe": {"rows": FI.PROBE_ROWS, "results": PROBE.to_dict("records")},
    "frozen_references": FROZEN.to_dict("records"),
    "stage3_dag": STAGE3_DAG,
    "accepted_artifacts": ACCEPTED,
    "duplicates": {"pairs": list(FI.DUPLICATE_PAIRS), "related": list(FI.RELATED_COPIES),
                   "callers": CALLERS, "parity": PARITY.to_dict("records"),
                   "binning_port": BINNING_PORT},
})
RECORD = RR.read_run_record(RUN.run_id, "feature_inventory")
print(f"record  : {RECORD_PATH.relative_to(BANK_ROOT)}")
print(f"content : {RECORD['content_sha256']}")

## Cell 14 — The PR 42 time index
The rows of `bank-additional-full.csv` are in date order, May 2008 to November 2010, with no year and no day of
month. `verify_source_order` checks that each training row's index label is its line in the CSV — through the
database load (`raw_bank` rowids in insertion order, `transform.sql` reading `ORDER BY rowid`), preprocessing
and the split — reading only the training rows' lines. `build_time_index` then derives, from training rows only:
the **period** (year inferred where the month decreases in source order; checked against the monthly indicators
and the calendar) and the **contact day** (a run of rows sharing year, month, weekday and the daily
`euribor3m`). The finest defensible resolution is the contact day: contacts of the same day, and when any
outcome became known, are unresolved (`EA.RESOLUTION`).

In [ ]:
# ============================================================
# CELL 14: TIME INDEX — source order, periods and contact days
# ============================================================
SOURCE_CHECKS = EA.verify_source_order(TRAIN.X_train, TRAIN.y_train)
TIME_INDEX = EA.build_time_index(TRAIN.X_train)
TIME_CHECKS = EA.time_index_checks(TRAIN.X_train, TIME_INDEX)
PERIODS = EA.period_table(TIME_INDEX, TRAIN.y_train)
print(EA.require(SOURCE_CHECKS, "source order"))
print(EA.require(TIME_CHECKS, "time index"))
RECURRING_DAY_KEYS = EA.recurring_day_keys(TIME_INDEX, TRAIN.X_train)
print(f"{TIME_INDEX['day'].nunique()} contact days in {len(PERIODS)} periods; "
      f"{RECURRING_DAY_KEYS} day keys recur in a later week of their month")
PERIODS

## Cell 15 — How the encodings are built
`EA.CONSTRUCTION` states, from the production code, how each audited encoding is fitted, applied and smoothed,
its fallbacks, and how a fit row enters its own value. On the full training split: each Stage 1 euribor bin's
rows, rate and the periods it spans, and each day x month cell's rows and the years it pools (the key has no year).

In [ ]:
# ============================================================
# CELL 15: ENCODING CONSTRUCTION
# ============================================================
ENCODINGS = pd.DataFrame(EA.CONSTRUCTION).T
STAGE1_BUILD = EA.stage1_construction(TRAIN.X_train, TRAIN.y_train, TIME_INDEX)
BINS, CELLS = STAGE1_BUILD["bins"], STAGE1_BUILD["cells"]
print(STAGE1_BUILD["settings"].to_string(index=False))
print(f"euribor bins spanning at most three periods: {(BINS['periods'] <= 3).sum()}/{len(BINS)}; "
      f"day x month cells pooling more than one year: {(CELLS['n_years'] > 1).sum()}/{len(CELLS)}")
BINS

## Cell 16 — Fitting paths
Every path that fits an audited encoding, as training positions, from the production fold plans: Stage 1
(both arms) final fit, 10 outer folds (tuning and OOF) and 10 x 10 inner calibration folds; Stage 2 (both
routers) final fit, its 5 label-excluded inner folds, 5 outer folds and their 5 x 5 label-excluded inner folds;
Stage 3 (both arms) final fit, 5 tuning, 10 reporting and 5 x 5 inner folds. The checks confirm each plan
partitions what it covers and where plans coincide (so evidence is shared); outer fold 0 and its inner plan are
compared with what the accepted Stage 2 run recorded.

In [ ]:
# ============================================================
# CELL 16: FITTING PATHS
# ============================================================
PATHS = EA.fit_paths(TRAIN.y_train)
PATH_TABLE = EA.path_table(PATHS)
PLAN_CHECKS = EA.plan_checks(TRAIN.y_train, PATHS)
ROUTER_PLAN = EA.router_plan_reference(TRAIN.X_train, TRAIN.y_train, PATHS)
print(EA.require(PLAN_CHECKS, "fold plans"))
print(EA.require(ROUTER_PLAN.rename(columns={"item": "check"}), "accepted Stage 2 fold 0 plan"))
PATH_TABLE

## Cell 17 — Fold isolation
Label flips through the production fold builders, each paired with a control that must change (so the check
can fail): Stage 1 `build_fold_designs` / `inner_fold_designs` (the calls both arms make), the Stage 2
label-excluded encoder, and one training row's own label in the full-train fit (mechanism a, measured). The
Stage 2 outer and Stage 3 fold fits are checked at the call sites: each is passed the fold's training labels only.
That the accepted fits used these refits is read from their records: the boundary extractors' fold checks (Cell 11
here, and the verified Commit 2 record, `RR.COMMIT2_INVENTORY`) and the accepted Stage 1 fits of both arms (PR 39
Commit 2 evidence; the MLP has no extractor).

The black-box MLP early-stops on a subset drawn **inside** each fit, after that fit's encoders were fitted on all of
its rows (`EA.MLP_EARLY_STOPPING_ORDER`). The subset's labels therefore enter the encodings of both the optimization
rows and the validation rows; a properly isolated split would exclude the whole subset from encoder fitting
(`EA.MLP_ISOLATED_SPLIT`). `mlp_early_stopping` reproduces the split on every Stage 1 fit (confirmed against the
estimator's own validation ROC-AUC) and measures the subset's share of both sides' label mass. Effects on the stopping
epoch, predictions and AUC are not quantified. Outer held-out isolation is a separate finding: those rows are outside
the subset, the encoders and the network, and are assessed by the flips above (`EA.MLP_OUTER_ISOLATION`).

In [ ]:
# ============================================================
# CELL 17: FOLD ISOLATION — label flips and in-fold label arguments
# ============================================================
COMMIT2 = RR.read_run_record(*RR.COMMIT2_INVENTORY)
ISOLATION = EA.isolation_checks(TRAIN.X_train, TRAIN.y_train, PATHS)
ACCEPTED_FITS = EA.accepted_fit_scopes(ACCEPTED, COMMIT2)
EARLY_STOPPING = EA.mlp_early_stopping(TRAIN.X_train, TRAIN.y_train, PATHS)
print(EA.require(ISOLATION, "fold isolation"))
print(EA.require(ACCEPTED_FITS.rename(columns={"item": "check"}), "accepted fits use fold-fitted encoders"))
print(EA.require(EARLY_STOPPING["checks"], "MLP early-stopping split reproduced"))
print(EARLY_STOPPING["summary"].round(5).to_string(index=False))
print(EARLY_STOPPING["finding"])
ISOLATION[["check", "passed", "detail"]]

## Cell 18 — Which labels reach each encoded value, and when
Every encoded value on every path is written as a combination of fit-row labels (bin or cell members, the
smoothing prior, the kernel's reference rows, the curvature's neighbouring bins), reconstructed and compared with
the production output, and its label mass is split by when each contributing contact happened relative to the
encoded row: earlier day, same day (unresolved), later day of the same month, later month. Roles: `held_out`
(a fold's held-out rows, each located among the fit's source rows so that its own label would be measured: b, and
c), `fit_rows` (the fit's own rows, i.e. the training representation: a, and its look-ahead) and `final_mapping`
(the final fits at each training row's contact day, standing for a new row contacted then; by convention its own
label counts as a same-day contact). Every comparison with production fails on a nonfinite value, and the cell
stops unless the completed comparisons match the expected ones (`EA.COMPARISON_POLICY`: the Stage 2 fold fits' own
rows are the only intentional skips) and no held-out encoding carries its own label.

In [ ]:
# ============================================================
# CELL 18: ATTRIBUTION BY FITTING PATH AND CONTACT TIMING
# ============================================================
AUDIT = EA.audit_paths(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, PATHS)
SUMMARY = EA.path_summary(AUDIT)
COVERAGE = EA.comparison_coverage(AUDIT, PATHS)
HELD_OUT_OWN = EA.held_out_own_labels(SUMMARY)
print(EA.require(COVERAGE, "reconstruction: expected versus completed comparisons"))
print(COVERAGE.iloc[-1]["check"], "->", COVERAGE.iloc[-1]["detail"])
print(COVERAGE.iloc[-2]["check"], "->", COVERAGE.iloc[-2]["detail"])
print(EA.require(HELD_OUT_OWN, "held-out own labels (mechanism b, measured)"))
print(f"largest |reconstructed - production| over passed comparisons: "
      f"{SUMMARY['max_abs_diff'].dropna().max():.1e} (tolerance {EA.TOLERANCE:.0e})")
SUMMARY[["feature", "stage", "path", "role", "fits", "rows", "self_rows", "later_month_rows",
         "any_later_rows", "later_share_mean", "later_share_p90", "majority_later_rows"]]

## Cell 19 — The final mappings by contact period
The final fits score new rows. At each training row's contact day and covariates, the mean share of the label mass
from later days of the same month and from later months, by period: what a contemporaneous new row's encoding is
built from under the random split.

In [ ]:
# ============================================================
# CELL 19: FINAL MAPPINGS BY PERIOD
# ============================================================
BY_PERIOD = EA.mapping_by_period(AUDIT)
BY_PERIOD.assign(later=BY_PERIOD["later_day_same_month"] + BY_PERIOD["later_month"]).pivot_table(
    index="period", columns=["stage", "feature"], values="later").round(3)

## Cell 20 — Provenance and findings
Where the constants and the frozen Stage 2 thresholds came from (read from the PR 39 Step 3A record), separately
from the labels a row's transformation uses, and the finding per feature, stage and fitting path: mechanisms
(a), (b) and (c) in counts with denominators (`FINDINGS`; `FAMILY_FINDINGS` per family). Interpretation, limits and evidence locations:
`research_logs/pr42/encoding_audit.md`.

In [ ]:
# ============================================================
# CELL 20: PROVENANCE AND FINDINGS
# ============================================================
PROVENANCE = EA.provenance_table()
FINDINGS = EA.findings_table(SUMMARY)
FAMILY_FINDINGS = EA.family_findings(SUMMARY)
print(PROVENANCE[["item", "provenance"]].to_string(index=False))
FINDINGS[["feature", "stage", "path", "fits", "a_self", "b_held_out", "c_rows", "c_later_month",
          "c_any_later_day", "c_later_share_mean", "c_later_majority", "same_day_share_mean"]]

## Cell 21 — Research record: encoding audit
The run's second record, written beside the first: `research_logs/pr42/<run_id>/encoding_audit.json` and its
`.sha256`, read back through both. It also holds the section-by-section comparison of this run's inventory record
with the verified Commit 2 record (read in Cell 17, never written). On the machine that wrote that record only
`duplicates.callers` should differ (this step's modules import the feature code); `accepted_artifacts`,
`training_frames` and the block-cache scheme digest also differ where the artifacts are absent, the platform
differs or the checkout's line endings differ.

In [ ]:
# ============================================================
# CELL 21: WRITE THE ENCODING AUDIT RECORD
# ============================================================
INVENTORY_VS_COMMIT2 = pd.DataFrame(RR.compare_records(RECORD, COMMIT2))
AUDIT_PATH = RR.write_run_record(RUN, "encoding_audit", {
    "scope": "PR 42 Commit 3 Step 3A: encoding construction and leakage audit (training side)",
    "question": EA.QUESTION,
    "mechanisms": dict(EA.MECHANISMS),
    "time_index": {"source_order": dict(EA.SOURCE_ORDER), "resolution": dict(EA.RESOLUTION),
                   "source_checks": SOURCE_CHECKS.to_dict("records"),
                   "checks": TIME_CHECKS.to_dict("records"), "periods": PERIODS.to_dict("records"),
                   "contact_days": int(TIME_INDEX["day"].nunique()),
                   "recurring_day_keys": RECURRING_DAY_KEYS},
    "construction": {"encodings": dict(EA.CONSTRUCTION), "descendants": dict(EA.DESCENDANTS),
                     "stage1_settings": STAGE1_BUILD["settings"].to_dict("records"),
                     "stage1_bins": BINS.to_dict("records"), "stage1_cells": CELLS.to_dict("records")},
    "fitting_paths": {"paths": PATH_TABLE.to_dict("records"),
                      "plan_checks": PLAN_CHECKS.to_dict("records"),
                      "accepted_stage2_fold0": ROUTER_PLAN.to_dict("records")},
    "isolation": {"checks": ISOLATION.to_dict("records"),
                  "accepted_fits": ACCEPTED_FITS.to_dict("records"),
                  "mlp_early_stopping": {
                      "order": EARLY_STOPPING["order"],
                      "isolated_split": EARLY_STOPPING["isolated_split"],
                      "outer_isolation": EARLY_STOPPING["outer_isolation"],
                      "finding": EARLY_STOPPING["finding"],
                      "val_fraction": EARLY_STOPPING["val_fraction"],
                      "random_state": EARLY_STOPPING["random_state"],
                      "checks": EARLY_STOPPING["checks"].to_dict("records"),
                      "summary": EARLY_STOPPING["summary"].to_dict("records")}},
    "attribution": {"relations": list(EA.RELATIONS), "later": list(EA.LATER),
                    "summary": SUMMARY.to_dict("records"),
                    "coverage": COVERAGE.to_dict("records"),
                    "held_out_own_labels": HELD_OUT_OWN.to_dict("records"),
                    "fits": EA.compact_fits(AUDIT),
                    "final_mapping_by_period": BY_PERIOD.to_dict("records")},
    "provenance": PROVENANCE.to_dict("records"),
    "findings": {"by_path": FINDINGS.to_dict("records"),
                 "by_family": FAMILY_FINDINGS.to_dict("records"),
                 "mlp_early_stopping": EARLY_STOPPING["finding"]},
    "commit2_inventory": {"reference": "/".join(RR.COMMIT2_INVENTORY),
                          "reference_content_sha256": COMMIT2["content_sha256"],
                          "this_run_content_sha256": RECORD["content_sha256"],
                          "sections": INVENTORY_VS_COMMIT2.to_dict("records")},
})
AUDIT_RECORD = RR.read_run_record(RUN.run_id, "encoding_audit")
print(f"record  : {AUDIT_PATH.relative_to(BANK_ROOT)}")
print(f"content : {AUDIT_RECORD['content_sha256']}")
print(f"inventory vs Commit 2: {INVENTORY_VS_COMMIT2.loc[~INVENTORY_VS_COMMIT2['same'], ['section', 'part']].values.tolist() or 'identical'}")

## Step 3B — Past-only encodings and the temporal robustness protocol
Step 3A found the accepted encodings fold-isolated (mechanism b absent) but looking ahead in time (mechanism c).
Step 3B compares, for Stage 1, protocols without it. A **past-only** encoding of a row contacted on day d uses
only labels that could have been known then: the production `LRFeatureEngineer` fitted on the context's rows
contacted before day d, less an outcome lag, so the bin edges, bin rates, median fallback, cell rates and global
prior all come from those rows. Same-day labels are never used. The accepted and past-only encodings are compared
with the accepted LR and MLP settings, on identical rows, under the accepted random 10-fold plan and under
chronological windows. The 2 × 2 on the windows' rows contrasts the two encoding protocols under both splits.
Each contrast is a difference of whole protocols: the past-only protocol removes same-day and later labels, but
also each training row's own label, fits its edges, rates and fallbacks on fewer rows and leaves the cold-start
rows out of its fits, and no contrast here isolates one of these. The module is `feature_research.temporal_validity`;
the review document is `research_logs/pr42/temporal_validity.md`.

## Cell 22 — Registration
`TV.REGISTRATION` states what is known when each row is scored, the rules (lag 0 is the primary candidate; lags 5
and 20 contact days and the monthly rule are outcome-lag sensitivities), the cold start and the history floor, the
fallbacks for unseen values, the chronological windows, the models, the comparisons and the contract criteria. Its
digest goes into the record. Version 1.0.0 was fixed before any result; 1.1.0 (`revisions`) was made after review of
those results: it adds the lag-consistent chronological arm and its regression, names every contrast as a protocol
difference and changes the Step 3A record check, and changes no 1.0.0 comparison. The windows come from the time
index: 2008-05..2008-08 train only, then consecutive windows of at least 1,500 rows.

In [ ]:
# ============================================================
# CELL 22: REGISTRATION — rules, cold start, windows, comparisons
# ============================================================
REGISTRATION_DIGEST = TV.registration_digest()
WINDOWS = TV.window_table(TIME_INDEX, TRAIN.y_train)
print(f"{TV.REGISTRATION['id']} {TV.REGISTRATION['version']}  sha256 {REGISTRATION_DIGEST}")
print(f"cold start: the first {TV.COLD_START_DAYS} contact days (plus the lag); every state used holds "
      f">= {TV.MIN_STATE_ROWS} rows and >= {TV.MIN_STATE_POSITIVES} positives")
print(pd.DataFrame([{"rule": k, **v} for k, v in TV.RULES.items()]).to_string(index=False))
WINDOWS

## Cell 23 — Predictive runs
`TV.evaluate` fits, per fold of the accepted plan (`build_fold_designs`, 10 folds, seed 42) and per chronological
window:

- **LR**, every encoding: accepted (the fold's designs; the window fit), past-only under each rule (each fold's
  training rows are its own context), and in the windows `past_frozen` (predicted rows encoded by the cutoff state)
  and `past_updating` (predicted rows encoded at their own contact day, the model frozen).
- **Outcome lag in the windows**, two ways. `past_updating` under lag 5, lag 20 or monthly keeps the lag-0 model,
  trained on every pre-cutoff outcome, and slows only the prediction-time encodings: a lag-0 model whose incoming
  features update more slowly. `lag_consistent` operates under the delay throughout: the scaler and the model are
  fitted on the training rows whose outcomes are known at the cutoff under the lag (`TV.known_at_cutoff`), with the
  lag's encodings. At lag 0 the two are the same model (checked).
- **MLP**, accepted and primary past-only encodings, each with the accepted early stopping (the subset's labels are
  inside the encodings and the scaler, Step 3A) and with an **isolated** split: the estimator's own split is drawn
  first, the encoders and scaler are fitted without the subset, and every fit checks that the estimator stopped on
  those rows.

Rows a rule cannot encode (cold start) are left out of its fits and of its comparisons. Takes several minutes.

In [ ]:
# ============================================================
# CELL 23: PREDICTIVE RUNS — random 10-fold plan and chronological windows
# ============================================================
EVALUATION = TV.evaluate(TRAIN.X_train, TRAIN.y_train, TIME_INDEX)
print(f"{len(EVALUATION['scores'])} protocols scored; {len(EVALUATION['notes'])} MLP fits")

## Cell 24 — The past-only states against production, and availability
Every state a rule uses is the production encoder fitted on that state's rows: `TV.verification` fits
`LRFeatureEngineer` on exactly those rows and requires identical values on the rows that use the state, including
the lagged and monthly uses where the fallbacks occur. Availability per rule and scope: the rows a rule encodes,
the cold start, the euribor median fallback (a value outside the bins of the rows seen so far), unseen day × month
cells, the history behind each state, and how often the values differ from the accepted encoding's.

In [ ]:
# ============================================================
# CELL 24: PAST-ONLY STATES VERSUS PRODUCTION, AND AVAILABILITY
# ============================================================
VERIFICATION = TV.verification(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, EVALUATION)
print(TV.require(VERIFICATION, "past-only states equal the production encoder"))
print(VERIFICATION[["context", "rule", "days", "states_compared", "rows_compared",
                    "fallback_values_compared", "max_abs_diff"]].to_string(index=False))
AVAILABILITY = TV.availability_tables(EVALUATION)
AVAILABILITY_CHECKS = TV.availability_checks(AVAILABILITY)
print(TV.require(AVAILABILITY_CHECKS, "state history floors"))
PRIMARY_ALL = EVALUATION["encodings"][f"all|{TV.PRIMARY}"]
AVAILABILITY_BY_PERIOD = TV.availability_by_period(PRIMARY_ALL, TIME_INDEX, PRIMARY_ALL["rows"])
AVAILABILITY[["scope", "rule", "rows", "available", "available_share", "euribor_fallback_rows",
              "cell_unseen_rows", "min_history_rows", "min_bins",
              "subscribe_rate_by_euribor_bin_rows_differ", "subscribe_rate_by_dow_month_rows_differ"]]

## Cell 25 — Changing an unavailable label cannot change an earlier encoding
For each rule, in the full-history context, a random fold's context and an isolated early-stopping context, at 40
contact days spread over the timeline: flipping every label unavailable to that day's encodings (the same day,
later days, inside the lag, the same period under the monthly rule) leaves them unchanged; flipping the same day's
labels alone leaves them unchanged; flipping the available labels changes them, so the check can fail. For
contrast, the accepted rule's encodings change when the same day's and later labels flip. Cell 23 adds the same
check on a deployed model's scores (first window), that the frozen predicted encodings equal the window fit's, and
the lag-consistent regression: in the first and the last window, flipping the pre-cutoff outcomes still unavailable
under lag 5 or lag 20 leaves the lag-consistent model (scaler, coefficients, intercept) and the deployed scores fixed
before those outcomes arrive unchanged, flipping known outcomes changes the model, and the same flip changes the
lag-0 model the prediction-time-only arm uses.

In [ ]:
# ============================================================
# CELL 25: INVARIANCE — flips of unavailable labels
# ============================================================
INVARIANCE = TV.invariance_checks(TRAIN.X_train, TRAIN.y_train, TIME_INDEX, TV.invariance_contexts(EVALUATION))
print(TV.require(INVARIANCE, "past-only encodings ignore unavailable labels"))
print(TV.require(EVALUATION["chrono_checks"], "chronological windows and deployed scores"))
INVARIANCE

## Cell 26 — Predictive comparisons: LR
First, the accepted protocol reproduces PR 39 (the mean fold ROC-AUC of the accepted LR and MLP on the accepted
designs). Then every registered comparison, paired DeLong on the rows both protocols scored (`b − a`, 95%
interval), pooled, per segment and as the mean over windows, and the 2 × 2 on the windows' rows: **A**
random/accepted, **B** random/past-only, **C** chronological/accepted, **D** chronological/past-only updating.
Protocol differences: past-only versus accepted, B − A and D − C; chronological versus random, C − A and D − B.
Each is a difference of whole protocols, consistent with but not isolating any one change, and an interval that
includes 0 leaves a difference unresolved rather than showing it absent. Pooled ROC-AUC mixes five models' score
levels across windows whose prevalence ranges from 7% to 49%; the unweighted mean over windows measures ranking
within them. Both are reported, and they can disagree in sign.

In [ ]:
# ============================================================
# CELL 26: PREDICTIVE COMPARISONS — LR
# ============================================================
REPRODUCTION = TV.reproduction_checks(EVALUATION)
print(TV.require(REPRODUCTION, "accepted protocols reproduce PR 39"))
PROTOCOLS = TV.protocol_table(EVALUATION)
COMPARED = TV.comparisons(EVALUATION)
FACTORIAL = {arm: TV.factorial(EVALUATION, arm) for arm in ("lr", "mlp")}
COLUMNS = ["question", "scope", "auc_a", "auc_b", "diff", "ci_low", "ci_high", "rows", "positives"]
SCOPES = ["pooled", "mean over windows"]
print(FACTORIAL["lr"]["cells"].drop(columns="protocol").round(4).to_string(index=False))
print(FACTORIAL["lr"]["effects"].round(4).to_string(index=False))
print(EVALUATION["lag_consistent_fits"].to_string(index=False))
COMPARED[(COMPARED["arm"] == "lr") & COMPARED["scope"].isin(SCOPES)][COLUMNS].round(4)

## Cell 27 — Predictive comparisons: MLP and its early stopping
The MLP's stopping epochs and best validation ROC-AUC under the accepted and the isolated early stopping, the
difference isolated − accepted (same encoding protocol and rows), and the 2 × 2 protocol differences with isolated
early stopping, so that none of them carries the Step 3A internal-validation issue.

In [ ]:
# ============================================================
# CELL 27: PREDICTIVE COMPARISONS — MLP, ACCEPTED VERSUS ISOLATED EARLY STOPPING
# ============================================================
EARLY_STOPPING_FITS = TV.early_stopping_table(EVALUATION)
print(EARLY_STOPPING_FITS.round(4).to_string(index=False))
print(FACTORIAL["mlp"]["cells"].drop(columns="protocol").round(4).to_string(index=False))
print(FACTORIAL["mlp"]["effects"].round(4).to_string(index=False))
COMPARED[(COMPARED["arm"] == "mlp") & COMPARED["scope"].isin(SCOPES)][COLUMNS].round(4)

## Cell 28 — Contract decisions and the Stage 2–3 families
The registered criteria applied to each candidate: realizable (Cell 25), values differing from the accepted
encoding's on at least 1% of rows, available for at least 95% of training rows. A past-only rule is a new feature
definition, so it needs its own versioned contract with distinct feature names; the accepted PR 39 contracts,
artifacts and records are unchanged. `TV.candidate_contract()` is the proposed contract (not registered). The
Stage 2–3 families are extended by construction and by the Step 3A attribution, not by experiment: the table says
which conclusions rest on direct evidence.

In [ ]:
# ============================================================
# CELL 28: CONTRACT DECISIONS AND THE STAGE 2-3 FAMILIES
# ============================================================
DECISIONS = TV.contract_decisions(AVAILABILITY[AVAILABILITY["scope"] == "random held-out rows"],
                                  INVARIANCE, COMPARED)
CANDIDATE_CONTRACT = TV.candidate_contract()
STAGE23 = TV.stage23_extension(SUMMARY)
print(DECISIONS.round(4).to_string(index=False))
print(f"proposed contract {CANDIDATE_CONTRACT['id']} {CANDIDATE_CONTRACT['version']}: "
      f"{CANDIDATE_CONTRACT['features']} (names {CANDIDATE_CONTRACT['fingerprint']}, definition "
      f"{CANDIDATE_CONTRACT['definition_fingerprint']}); accepted {CANDIDATE_CONTRACT['accepted_contract']}")
STAGE23

## Cell 29 — The PR 49 protocol and the interim finding
`TV.PR49_PROTOCOL` is the design PR 49 executes: the chronological split, what is fitted at each cutoff and what
may update after it, how prevalence shift is reported for calibration, thresholds, abstention and the Stage 2
obligations, and how a missed obligation is reported. `TV.stage2_prevalence_projection` shows what a pure prior
shift alone does to the frozen routing's obligations: the accepted routers' training OOF class-conditional
routing rates (PR 38 report, no artifact read) held fixed, the prevalence moved to each window's, the projected
counts observed with `observe_routing` against the frozen contract, exactly as PR 49 reports a window.
`TV.interim_finding` states what PR 43 may claim from the frozen PR 39 evidence, what PRs 44–45 need and what
stays open for PR 49, with this run's numbers.

In [ ]:
# ============================================================
# CELL 29: PR 49 PROTOCOL AND INTERIM FINDING
# ============================================================
PREVALENCES = dict(zip(WINDOWS["segment"], WINDOWS["prevalence"]))
PROJECTION = TV.stage2_prevalence_projection(PREVALENCES)
INTERIM = TV.interim_finding(COMPARED, FACTORIAL, DECISIONS, PROJECTION)
for key in ("split", "updated_after_cutoff", "prevalence_shift", "prior_shift_attribution",
            "missed_obligation_report"):
    print(f"{key}: {TV.PR49_PROTOCOL[key]}")
print(PROJECTION["bands"].round(4).to_string(index=False))
for key in ("finding", "pr43", "pr44_45", "pr49_open"):
    print(f"\n{key}: {INTERIM[key]}")
for key, text in INTERIM["interpretation"].items():
    print(f"\ninterpretation, {key}: {text}")
PROJECTION["projection"].round(4)

## Cell 30 — Research record: temporal validity
The run's third record, beside the first two: `research_logs/pr42/<run_id>/temporal_validity.json` and its
`.sha256`, read back through both. First, this run's Step 3A records are compared with the accepted Step 3A run
(`TV.STEP3A_RECORD`, `c3_20261122_201830`), and the applicable checks are enforced **before** the record is
written. If that run's directory is absent, the comparison is reported as unavailable. If it exists, it must hold
both records and their sidecars; a missing file fails the integrity check in any environment. Each record's
sidecar and content digest are checked, and that the digest is the accepted one. Exact reproduction is required
only where this run matches the accepted run's environment (operating system, platform, Python, interpreter and
package versions) and the same accepted artifacts were available to both inventories; on any other checkout, such
as Linux after the accepted run is committed, it is reported with the reason and not required. The accepted run is
read, never written.

In [ ]:
# ============================================================
# CELL 30: WRITE THE TEMPORAL VALIDITY RECORD
# ============================================================
STEP3A = TV.step3a_reproduction({"feature_inventory": RECORD, "encoding_audit": AUDIT_RECORD},
                                RUN.environment)
STEP3A_CHECKS = TV.step3a_checks(STEP3A)
print(STEP3A.drop(columns=["accepted_content_sha256"]).to_string(index=False))
print(TV.require(STEP3A_CHECKS, "accepted Step 3A record (integrity; reproduction where required)"))
TEMPORAL_PATH = RR.write_run_record(RUN, "temporal_validity", {
    "scope": "PR 42 Commit 3 Step 3B: past-only Stage 1 encodings and the temporal robustness protocol",
    "registration": {"content": TV.REGISTRATION, "sha256": REGISTRATION_DIGEST,
                     "rules": TV.RULES, "cold_start_days": TV.COLD_START_DAYS,
                     "min_state_rows": TV.MIN_STATE_ROWS, "min_state_positives": TV.MIN_STATE_POSITIVES,
                     "windows": WINDOWS.to_dict("records")},
    "settings": {k: v for k, v in EVALUATION["settings"].items()},
    "verification": VERIFICATION.to_dict("records"),
    "availability": {"tables": AVAILABILITY.to_dict("records"),
                     "checks": AVAILABILITY_CHECKS.to_dict("records"),
                     "primary_by_period": AVAILABILITY_BY_PERIOD.to_dict("records")},
    "invariance": {"encodings": INVARIANCE.to_dict("records"),
                   "chronological": EVALUATION["chrono_checks"].to_dict("records")},
    "predictive": {"reproduction": REPRODUCTION.to_dict("records"),
                   "protocols": PROTOCOLS.to_dict("records"),
                   "comparisons": COMPARED.to_dict("records"),
                   "factorial": {arm: {"cells": f["cells"].to_dict("records"),
                                       "effects": f["effects"].to_dict("records"),
                                       "rows": f.get("rows"), "positives": f.get("positives"),
                                       "windows": f.get("windows")} for arm, f in FACTORIAL.items()},
                   "fold_roc_auc": EVALUATION["fold_roc_auc"],
                   "lag_consistent_fits": EVALUATION["lag_consistent_fits"].to_dict("records"),
                   "mlp_fits": EVALUATION["notes"].to_dict("records"),
                   "mlp_early_stopping": EARLY_STOPPING_FITS.to_dict("records")},
    "contracts": {"decisions": DECISIONS.to_dict("records"), "candidate": CANDIDATE_CONTRACT},
    "stage23_extension": STAGE23.to_dict("records"),
    "pr49_protocol": TV.PR49_PROTOCOL,
    "stage2_prior_shift": {"contract": PROJECTION["contract"], "routing": PROJECTION["routing"],
                           "nominal_rows": PROJECTION["nominal_rows"],
                           "bands": PROJECTION["bands"].to_dict("records"),
                           "projection": PROJECTION["projection"].to_dict("records")},
    "interim_finding": INTERIM,
    "step3a_reproduction": {"accepted": TV.STEP3A_RECORD, "environment_keys": list(TV.ENVIRONMENT_KEYS),
                            "comparison": STEP3A.to_dict("records"),
                            "checks": STEP3A_CHECKS.to_dict("records")},
})
TEMPORAL_RECORD = RR.read_run_record(RUN.run_id, "temporal_validity")
print(f"record  : {TEMPORAL_PATH.relative_to(BANK_ROOT)}")
print(f"content : {TEMPORAL_RECORD['content_sha256']}")

## Later PR 42 commits
Sections added by later commits, on the same training side and record conventions:

| Commit | Section | Sources |
|---|---|---|
| 3 | Step 3A (Cells 14–21): time index, encoding construction, fitting paths, fold isolation, attribution by contact timing, provenance, findings. Step 3B (Cells 22–30): past-only Stage 1 encodings, availability, invariance, predictive effect (encoding versus split, MLP early stopping), contract decisions, the PR 49 protocol and the interim finding | Cells 2, 6–8 here; `shared.stage1.crossfit`, `stage2_crossfit`, `shared.stage3.data` |
| 4 | Stage 2 NSD/ECI binning drift, sensitivity and candidates | E 15A–15B; `stage2_binning`, `stage2_crossfit` |
| 5 | Feature analysis, interactions and EBM interaction selection | E 7–10; G 7 |
| 6 | Feature-contract recommendations and final validation | this notebook's records |